# WTI Producer Hedge Simulator

This notebook asks one main question: how much can a crude producer reduce revenue risk by using WTI futures?

The producer owns future oil production. If oil prices fall before that oil is sold, revenue falls too. A short futures hedge can help offset part of that loss.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import yfinance as yf
from pandas_datareader import data as web

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

from src.hedge_engine import HedgeAssumptions, prepare_monthly_market_data, compare_hedge_ratios, stress_test
from src.basis_risk import compare_basis_scenarios, simulate_basis_scenario, BasisHedgeAssumptions
from src.min_variance import minimum_variance_hedge_ratio, hedge_ratio_diagnostics, rolling_minimum_variance_hedge_ratio, contracts_for_hedge_ratio


## Load the WTI prices

I use WTI spot prices for the physical side and WTI futures prices for the hedge.

In [ ]:
spot = web.DataReader('DCOILWTICO', 'fred', '2018-01-01')['DCOILWTICO']
raw = yf.download('CL=F', start='2018-01-01', auto_adjust=False, progress=False)
futures = raw['Close'].iloc[:, 0] if isinstance(raw.columns, pd.MultiIndex) else raw['Close']
market = prepare_monthly_market_data(spot, futures)
market.tail()


## Compare different hedge sizes

The producer expects to sell 100,000 barrels per month. Here I compare what happens when none, some, or all of that production is hedged.

In [ ]:
assumptions = HedgeAssumptions(monthly_production_bbl=100_000)
summary, simulations = compare_hedge_ratios(market, assumptions=assumptions)
summary


## Stress test: what if oil falls 25%?

This checks how a 75% hedge changes the result during a large drop in crude prices.

In [ ]:
stress_test(spot_price=75, futures_entry=76, spot_shock_pct=-0.25, hedge_ratio=0.75, assumptions=assumptions)


## Midland vs. Cushing basis risk

The producer may sell crude in Midland while using futures tied to Cushing. Those prices usually move together, but not perfectly.

This section shows what happens when the difference between them moves against the producer.

In [ ]:
basis_scenarios = compare_basis_scenarios(
    realized_basis_values=(1, 0, -1, -3, -5, -10),
    basis_hedge_ratios=(0, 0.50, 1.00),
    cushing_futures_entry=75,
    cushing_futures_exit=60,
    cushing_spot_exit=60,
    locked_basis_per_bbl=-1,
    flat_price_hedge_ratio=1.0,
    assumptions=assumptions,
)
basis_scenarios[[
    'basis_hedge_ratio',
    'realized_midland_basis',
    'basis_swap_pnl',
    'residual_revenue_risk',
]]


### What this means

A WTI futures hedge can reduce the main oil-price risk and still leave location risk. If Midland gets cheaper relative to Cushing, the producer can still receive less than expected.

## Let the historical data choose a hedge size

Here I use the relationship between spot and futures price changes to estimate the hedge size that reduced price movement the most.

In [ ]:
mv_ratio = minimum_variance_hedge_ratio(market)
mv_contracts = contracts_for_hedge_ratio(100_000, mv_ratio)
mv_ratio, mv_contracts


In [ ]:
hedge_ratio_diagnostics(market, mv_contracts / 100)


In [ ]:
rolling_mv = rolling_minimum_variance_hedge_ratio(market, window=24)
rolling_mv.tail()
